In [1]:
# ============================================================
# PHASE 2 - Data Cleaning + Feature Engineering
# ============================================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

BASE_DIR    = r"C:\Users\yipch\ecommerce-analytics-portfolio"
SAMPLE_DIR  = os.path.join(BASE_DIR, "data", "samples")
CLEAN_DIR   = os.path.join(BASE_DIR, "data", "processed")

# Load the saved main sample from Phase 1
df = pd.read_csv(
    os.path.join(SAMPLE_DIR, "sampled_main.csv"),
    parse_dates=["event_time"]
)

print(f"✅ Loaded main dataset")
print(f"   Shape   : {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"   Columns : {list(df.columns)}")

✅ Loaded main dataset
   Shape   : 1,610,402 rows × 10 cols
   Columns : ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session', 'month']


In [2]:
# ============================================================
# STEP 2 - Pre-Cleaning Snapshot (before vs after comparison)
# ============================================================

def snapshot(df, label=""):
    print(f"\n{'='*55}")
    print(f"  SNAPSHOT : {label}")
    print(f"{'='*55}")
    print(f"  Rows            : {df.shape[0]:,}")
    print(f"  Duplicate rows  : {df.duplicated().sum():,}")
    print(f"\n  Missing values:")
    missing = df.isnull().sum()
    for col, cnt in missing.items():
        pct = cnt / len(df) * 100
        flag = "⚠️ " if pct > 0 else "✅"
        print(f"    {flag} {col:<20} : {cnt:>7,}  ({pct:.1f}%)")
    print(f"\n  Price issues:")
    print(f"    Zero price      : {(df['price'] == 0).sum():,}")
    print(f"    Negative price  : {(df['price'] < 0).sum():,}")
    print(f"    Price > 10,000  : {(df['price'] > 10000).sum():,}")

snapshot(df, "BEFORE CLEANING")


  SNAPSHOT : BEFORE CLEANING
  Rows            : 1,610,402
  Duplicate rows  : 1,709

  Missing values:
    ✅ event_time           :       0  (0.0%)
    ✅ event_type           :       0  (0.0%)
    ✅ product_id           :       0  (0.0%)
    ✅ category_id          :       0  (0.0%)
    ⚠️  category_code        : 517,940  (32.2%)
    ⚠️  brand                : 222,747  (13.8%)
    ✅ price                :       0  (0.0%)
    ✅ user_id              :       0  (0.0%)
    ✅ user_session         :       0  (0.0%)
    ✅ month                :       0  (0.0%)

  Price issues:
    Zero price      : 3,591
    Negative price  : 0
    Price > 10,000  : 0


In [3]:
# ============================================================
# STEP 3A - Remove duplicate rows
# ============================================================

before = len(df)
df = df.drop_duplicates()
after  = len(df)

print(f"✅ Duplicates removed : {before - after:,} rows dropped")
print(f"   Rows remaining     : {after:,}")

✅ Duplicates removed : 1,709 rows dropped
   Rows remaining     : 1,608,693


In [4]:
# ============================================================
# STEP 3B - Fix missing values
# ============================================================

# category_code: fill with "unknown" (not all products are categorized)
df["category_code"] = df["category_code"].fillna("unknown")

# brand: fill with "unknown"
df["brand"] = df["brand"].fillna("unknown")

# Drop rows where critical fields are null
critical_cols = ["event_time", "event_type", "user_id",
                 "product_id", "price"]
before = len(df)
df = df.dropna(subset=critical_cols)
after  = len(df)

print(f"✅ Missing values handled")
print(f"   category_code → filled with 'unknown'")
print(f"   brand         → filled with 'unknown'")
print(f"   Critical nulls dropped : {before - after:,} rows")

✅ Missing values handled
   category_code → filled with 'unknown'
   brand         → filled with 'unknown'
   Critical nulls dropped : 0 rows


In [5]:
# ============================================================
# STEP 3C - Fix price issues
# ============================================================

before = len(df)

# Remove zero and negative prices
df = df[df["price"] > 0]

# Remove extreme outliers (price > 10,000 = likely data error)
df = df[df["price"] <= 10_000]

after = len(df)

print(f"✅ Price issues fixed")
print(f"   Rows removed : {before - after:,}")
print(f"   Valid price range : ${df['price'].min():.2f} — ${df['price'].max():.2f}")
print(f"   Median price      : ${df['price'].median():.2f}")

✅ Price issues fixed
   Rows removed : 3,591
   Valid price range : $0.77 — $2574.07
   Median price      : $165.51


In [6]:
# ============================================================
# STEP 3D - Standardize text columns
# ============================================================

df["event_type"]     = df["event_type"].str.strip().str.lower()
df["brand"]          = df["brand"].str.strip().str.lower()
df["category_code"]  = df["category_code"].str.strip().str.lower()

# Validate event_type — keep only known values
valid_events = ["view", "cart", "purchase"]
before = len(df)
df = df[df["event_type"].isin(valid_events)]
after  = len(df)

print(f"✅ Text columns standardized (stripped + lowercased)")
print(f"   Invalid event_types dropped : {before - after:,}")
print(f"   Valid event_types : {df['event_type'].unique()}")

✅ Text columns standardized (stripped + lowercased)
   Invalid event_types dropped : 0
   Valid event_types : <ArrowStringArray>
['view', 'cart', 'purchase']
Length: 3, dtype: str


In [7]:
# ============================================================
# STEP 4A - DateTime features
# Unlocks: hourly trends, day-of-week patterns, weekly cohorts
# ============================================================

df["event_date"]    = df["event_time"].dt.date
df["event_hour"]    = df["event_time"].dt.hour
df["event_dow"]     = df["event_time"].dt.day_name()     # Monday, Tuesday...
df["event_dow_num"] = df["event_time"].dt.dayofweek      # 0=Mon, 6=Sun
df["event_week"]    = df["event_time"].dt.isocalendar().week.astype(int)
df["event_month"]   = df["event_time"].dt.to_period("M").astype(str)

print("✅ DateTime features created:")
print("   event_date    → YYYY-MM-DD")
print("   event_hour    → 0–23 (for peak hour analysis)")
print("   event_dow     → day name (for weekday patterns)")
print("   event_dow_num → 0–6 (for sorting)")
print("   event_week    → ISO week number (for cohorts)")
print("   event_month   → YYYY-MM (for monthly trends)")
print(f"\n   Sample:")
print(df[["event_time","event_date","event_hour",
          "event_dow","event_week","event_month"]].head(3).to_string())

✅ DateTime features created:
   event_date    → YYYY-MM-DD
   event_hour    → 0–23 (for peak hour analysis)
   event_dow     → day name (for weekday patterns)
   event_dow_num → 0–6 (for sorting)
   event_week    → ISO week number (for cohorts)
   event_month   → YYYY-MM (for monthly trends)

   Sample:
                 event_time  event_date  event_hour event_dow  event_week event_month
0 2019-10-01 00:01:39+00:00  2019-10-01           0   Tuesday          40     2019-10
1 2019-10-01 00:01:54+00:00  2019-10-01           0   Tuesday          40     2019-10
2 2019-10-01 00:02:15+00:00  2019-10-01           0   Tuesday          40     2019-10


In [8]:
# ============================================================
# STEP 4B - Category hierarchy split
# category_code format: "electronics.smartphone"
# Unlocks: top-level category analysis vs sub-category
# ============================================================

df["category_l1"] = df["category_code"].apply(
    lambda x: x.split(".")[0] if x != "unknown" else "unknown"
)
df["category_l2"] = df["category_code"].apply(
    lambda x: x.split(".")[1] if (x != "unknown" and "." in x) else "unknown"
)

print("✅ Category hierarchy split:")
print(f"   category_l1 (top level) — {df['category_l1'].nunique()} unique:")
top_cats = df["category_l1"].value_counts().head(8)
for cat, cnt in top_cats.items():
    print(f"     {cat:<30} : {cnt:,}")

print(f"\n   category_l2 (sub level) — {df['category_l2'].nunique()} unique")

✅ Category hierarchy split:
   category_l1 (top level) — 14 unique:
     electronics                    : 588,256
     unknown                        : 515,807
     appliances                     : 194,573
     computers                      : 99,146
     apparel                        : 65,433
     furniture                      : 47,726
     auto                           : 32,117
     construction                   : 25,434

   category_l2 (sub level) — 59 unique


In [9]:
# ============================================================
# STEP 4C - Revenue column
# Only purchases have real revenue (views/carts = $0)
# Unlocks: revenue trend, product revenue, brand revenue
# ============================================================

df["revenue"] = np.where(df["event_type"] == "purchase", df["price"], 0)

total_rev = df["revenue"].sum()
n_orders  = (df["event_type"] == "purchase").sum()

print(f"✅ Revenue column created")
print(f"   Total revenue in sample : ${total_rev:,.2f}")
print(f"   Total purchase events   : {n_orders:,}")
print(f"   Avg order value (AOV)   : ${total_rev/n_orders:.2f}" if n_orders > 0 else "   No purchases found")

✅ Revenue column created
   Total revenue in sample : $7,416,954.89
   Total purchase events   : 24,602
   Avg order value (AOV)   : $301.48


In [10]:
    # ============================================================
# STEP 4D - Session-level features
# Unlocks: session length, session conversion flag
# ============================================================

# Sort for session calculations
df = df.sort_values(["user_session", "event_time"]).reset_index(drop=True)

# Flag: did this session result in a purchase?
sessions_with_purchase = set(
    df[df["event_type"] == "purchase"]["user_session"]
)
df["session_converted"] = df["user_session"].isin(sessions_with_purchase).astype(int)

# Count events per session
session_event_counts = df.groupby("user_session")["event_type"].transform("count")
df["session_event_count"] = session_event_counts

conversion_rate = df.drop_duplicates("user_session")["session_converted"].mean()

print(f"✅ Session features created:")
print(f"   session_converted    → 1 if session had a purchase, else 0")
print(f"   session_event_count  → how many events in that session")
print(f"\n   Session conversion rate : {conversion_rate:.2%}")
print(f"   Avg events per session  : {df['session_event_count'].mean():.1f}")

✅ Session features created:
   session_converted    → 1 if session had a purchase, else 0
   session_event_count  → how many events in that session

   Session conversion rate : 6.14%
   Avg events per session  : 15.6


In [11]:
# ============================================================
# STEP 4E - User-level purchase flag
# Unlocks: conversion analysis, buyer vs browser segmentation
# ============================================================

users_who_purchased = set(df[df["event_type"] == "purchase"]["user_id"])
df["user_has_purchased"] = df["user_id"].isin(users_who_purchased).astype(int)

buyers     = df[df["user_has_purchased"] == 1]["user_id"].nunique()
non_buyers = df[df["user_has_purchased"] == 0]["user_id"].nunique()
total_u    = df["user_id"].nunique()

print(f"✅ User purchase flag created:")
print(f"   Buyers     : {buyers:,}  ({buyers/total_u:.1%} of users)")
print(f"   Non-buyers : {non_buyers:,}  ({non_buyers/total_u:.1%} of users)")

✅ User purchase flag created:
   Buyers     : 11,698  (11.7% of users)
   Non-buyers : 87,960  (88.3% of users)


In [12]:
# ============================================================
# STEP 5 - Post-Cleaning Snapshot
# ============================================================

snapshot(df, "AFTER CLEANING")

print(f"\n📋 Final columns ({df.shape[1]} total):")
for col in df.columns:
    print(f"   {col:<25} : {df[col].dtype}")


  SNAPSHOT : AFTER CLEANING
  Rows            : 1,605,102
  Duplicate rows  : 0

  Missing values:
    ✅ event_time           :       0  (0.0%)
    ✅ event_type           :       0  (0.0%)
    ✅ product_id           :       0  (0.0%)
    ✅ category_id          :       0  (0.0%)
    ✅ category_code        :       0  (0.0%)
    ✅ brand                :       0  (0.0%)
    ✅ price                :       0  (0.0%)
    ✅ user_id              :       0  (0.0%)
    ✅ user_session         :       0  (0.0%)
    ✅ month                :       0  (0.0%)
    ✅ event_date           :       0  (0.0%)
    ✅ event_hour           :       0  (0.0%)
    ✅ event_dow            :       0  (0.0%)
    ✅ event_dow_num        :       0  (0.0%)
    ✅ event_week           :       0  (0.0%)
    ✅ event_month          :       0  (0.0%)
    ✅ category_l1          :       0  (0.0%)
    ✅ category_l2          :       0  (0.0%)
    ✅ revenue              :       0  (0.0%)
    ✅ session_converted    :       0  (0.0%)


In [13]:
# ============================================================
# STEP 6 - Save clean dataset
# ============================================================

output_path = os.path.join(CLEAN_DIR, "clean_events.csv")
df.to_csv(output_path, index=False)

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"✅ Clean dataset saved!")
print(f"   Path    : {output_path}")
print(f"   Size    : {size_mb:.1f} MB")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")

print(f"\n🎯 This file is used for ALL future phases (SQL, Power BI, Streamlit)")

✅ Clean dataset saved!
   Path    : C:\Users\yipch\ecommerce-analytics-portfolio\data\processed\clean_events.csv
   Size    : 325.6 MB
   Rows    : 1,605,102
   Columns : 22

🎯 This file is used for ALL future phases (SQL, Power BI, Streamlit)
